# Search Service Evaluation

This notebook checks the pass-only Search Service as a working system. It reviews artifact health, compares retrieval variants, inspects ranking behavior, and evaluates the default path against manually judged dev relevance labels.

The public default uses two-worker retrieval parallelism. Controlled component experiments below use a separate serial service so ranking changes are not mixed with local scheduling effects.

In [1]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 240)
sys.path.append("..")

from src.real_estate_nlp.search_service import SearchService

---

## 1. Artifacts Loading

Load the active pass-only snapshot through the service entry point. The default service uses two retrieval workers. Snapshot validation runs during loading, so a missing file, checksum mismatch, or non-pass listing in a public artifact stops the notebook here. Dense and Cross Encoder warm-up costs are recorded separately; the remaining tables report warm query latency.

In [2]:
search_root = "../data/models/search"
service = SearchService.from_active_snapshot(search_root=search_root)
cold_start = service.warm_up(include_cross_encoder=True)

manifest = service.snapshot.manifest
catalog = pd.DataFrame(service.snapshot.catalog_by_id.values())

snapshot_profile = pd.DataFrame([
    {
        "snapshot_id": service.snapshot.snapshot_id,
        "source_listings": manifest["source_listing_count"],
        "pass_listings": manifest["public_listing_count"],
        "retrievable_listings": manifest["retrievable_listing_count"],
        "pass_rate": manifest["public_listing_count"] / manifest["source_listing_count"],
        "rule_version": manifest["compliance_rule_version"],
        "dense_model": manifest["dense_model"],
    }
])
snapshot_profile


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

,snapshot_id,source_listings,pass_listings,retrievable_listings,pass_rate,rule_version,dense_model
0,20260903T124929Z_5466f5d6de,53122,53091,52763,0.999416,federal-1.1,sentence-transformers/all-MiniLM-L6-v2


In [3]:
artifact_counts = pd.DataFrame([
    {"artifact": "catalog", "listings": len(service.snapshot.catalog_by_id)},
    {"artifact": "summaries", "listings": len(service.snapshot.summaries_by_id)},
    {"artifact": "signals", "listings": len(service.snapshot.signals.signals_by_listing_id)},
    {"artifact": "dense index", "listings": len(service.snapshot.semantic.metadata)},
    {"artifact": "BM25 index", "listings": len(service.snapshot.bm25.metadata)},
    {"artifact": "textless pass listings", "listings": len(service.snapshot.pass_listing_ids - service.snapshot.retrievable_listing_ids)},
])
artifact_counts


,artifact,listings
0,catalog,53091
1,summaries,53091
2,signals,53091
3,dense index,52763
4,BM25 index,52763
5,textless pass listings,328


The catalog, summaries, and signals should all cover the pass-only set. Dense and BM25 intentionally exclude listings without usable remark text. Those listings are only added as a structured fallback when no meaningful positive soft requirement is present.


---

## 2. Query Suite

These queries exercise structured filters, softer listing preferences, keyword-heavy language, and a strict price-sort request. They are diagnostic examples, not relevance labels.


In [4]:
query_suite = pd.DataFrame([
    {"scenario": "structured plus amenity", "query": "Find 3 bedroom homes in Galt with a pool"},
    {"scenario": "amenity retrieval", "query": "Find homes in Los Angeles with a pool"},
    {"scenario": "condition and budget", "query": "Updated homes in Riverside under $750k"},
    {"scenario": "property type and layout", "query": "Single family homes in Irvine with an open floor plan"},
    {"scenario": "location features", "query": "Homes near schools in Corona"},
    {"scenario": "strict price sort", "query": "Show me the cheapest 3 bedroom homes in Galt"},
])
query_suite


,scenario,query
0,structured plus amenity,Find 3 bedroom homes in Galt with a pool
1,amenity retrieval,Find homes in Los Angeles with a pool
2,condition and budget,Updated homes in Riverside under $750k
3,property type and layout,Single family homes in Irvine with an open floor plan
4,location features,Homes near schools in Corona
5,strict price sort,Show me the cheapest 3 bedroom homes in Galt


---

## 3. Default Relevance Search

For relevance requests, the external default fuses dense retrieval, BM25, and signal retrieval with RRF, then reranks the top 50 candidates with the Cross Encoder. Explicit price sorts retain the same variant label but skip text retrieval and reranking so the requested field order remains stable. Cold start is reported separately below; this table records warm end-to-end latency and source-level timings for each query.


In [5]:
default_results = {}
default_rows = []

for item in query_suite.to_dict("records"):
    result = service.search(item["query"], top_k=5)
    default_results[item["scenario"]] = result
    default_rows.append({
        "scenario": item["scenario"],
        "variant": result["meta"].get("variant"),
        "retrieval_mode": result["meta"].get("retrieval_execution"),
        "workers": service.retrieval_workers,
        "effective_sort": result["meta"].get("effective_sort"),
        "eligible_listings": result["meta"].get("eligible_count"),
        "returned": len(result["results"]),
        "signal_results": sum(row["signal_status"] == "selected" for row in result["results"]),
        "warm_latency_ms": result["meta"]["timings_ms"].get("total"),
        "hard_filter_ms": result["meta"]["timings_ms"].get("hard_filter"),
        "dense_ms": result["meta"]["timings_ms"].get("dense"),
        "bm25_ms": result["meta"]["timings_ms"].get("bm25"),
        "signals_ms": result["meta"]["timings_ms"].get("signals"),
        "retrieval_wall_ms": result["meta"]["timings_ms"].get("retrieval_wall"),
        "rrf_fusion_ms": result["meta"]["timings_ms"].get("rrf_fusion"),
        "signal_selection": result["meta"].get("signal_selection", {}).get("strategy"),
        "rrf_candidates": result["meta"].get("candidate_counts", {}).get("rrf_union"),
        "rerank_window": result["meta"].get("candidate_counts", {}).get("rerank_window"),
        "degraded_components": ", ".join(result["meta"]["degraded_components"]) or "none",
        "timings": result["meta"]["timings_ms"],
    })

default_summary = pd.DataFrame(default_rows)
cold_start_summary = pd.DataFrame([{
    "cold_start_dense_ms": cold_start["timings_ms"]["dense_warm_up"],
    "cold_start_cross_encoder_ms": cold_start["timings_ms"].get("cross_encoder_cold_start"),
}])
display(cold_start_summary)
display(default_summary)

,cold_start_dense_ms,cold_start_cross_encoder_ms
0,4249.56,2017.11


,scenario,variant,retrieval_mode,workers,effective_sort,eligible_listings,returned,signal_results,warm_latency_ms,hard_filter_ms,dense_ms,bm25_ms,signals_ms,retrieval_wall_ms,rrf_fusion_ms,signal_selection,rrf_candidates,rerank_window,degraded_components,timings
0,structured plus amenity,hybrid_cross_encoder,parallel,2,relevance,5,5,0,112.34,19.29,20.40,1.11,4.24,20.59,0.02,no_query_signals,5.0,5.0,none,"{'hard_filter': 19.29, 'dense': 20.4, 'bm25': 1.11, 'signals': 4.24, 'retrieval_wall': 20.59, 'rrf_fusion': 0.02, 'cross_encoder_rerank': 37.47, 'total': 112.34}"
1,amenity retrieval,hybrid_cross_encoder,parallel,2,relevance,3441,5,5,292.42,15.99,41.71,13.97,8.38,55.54,0.44,dense_secondary,342.0,50.0,none,"{'hard_filter': 15.99, 'dense': 41.71, 'bm25': 13.97, 'signals': 8.38, 'retrieval_wall': 55.54, 'rrf_fusion': 0.44, 'cross_encoder_rerank': 196.99, 'total': 292.42}"
2,condition and budget,hybrid_cross_encoder,parallel,2,relevance,263,5,5,275.03,11.77,14.87,4.78,3.19,15.34,0.45,all_matches,183.0,50.0,none,"{'hard_filter': 11.77, 'dense': 14.87, 'bm25': 4.78, 'signals': 3.19, 'retrieval_wall': 15.34, 'rrf_fusion': 0.45, 'cross_encoder_rerank': 200.59, 'total': 275.03}"
3,property type and layout,hybrid_cross_encoder,parallel,2,relevance,746,5,5,232.15,12.32,16.49,6.39,3.96,19.36,0.38,dense_secondary,283.0,50.0,none,"{'hard_filter': 12.32, 'dense': 16.49, 'bm25': 6.39, 'signals': 3.96, 'retrieval_wall': 19.36, 'rrf_fusion': 0.38, 'cross_encoder_rerank': 151.34, 'total': 232.15}"
4,location features,hybrid_cross_encoder,parallel,2,relevance,410,5,5,273.30,9.43,34.75,4.96,1.32,35.20,0.25,all_matches,185.0,50.0,none,"{'hard_filter': 9.43, 'dense': 34.75, 'bm25': 4.96, 'signals': 1.32, 'retrieval_wall': 35.2, 'rrf_fusion': 0.25, 'cross_encoder_rerank': 188.88, 'total': 273.3}"
5,strict price sort,hybrid_cross_encoder,None,2,price_asc,5,5,0,69.14,9.69,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,none,"{'hard_filter': 9.69, 'total': 69.14}"


In [6]:
def signal_display(item):
    status = item.get("signal_status")
    if status == "not_used":
        return "not used"
    if status == "not_selected":
        return "not selected by signal source"
    if status == "no_match":
        return "no extracted match"
    return ", ".join(match["value"] for match in item["matched_signals"])


def compact_results(result, scenario):
    rows = []
    for item in result["results"]:
        rows.append({
            "scenario": scenario,
            "rank": item["rank"],
            "listing_id": item["listing_id"],
            "city": item["city"],
            "price": item["price"],
            "score": item.get("score"),
            "source_ranks": item["source_ranks"],
            "signal_match": signal_display(item),
            "match_tier": item.get("match_tier"),
            "retrieval_status": item["retrieval_status"],
            "retrieval_evidence": item["retrieval_evidence"],
            "summary": item["summary"],
        })
    return pd.DataFrame(rows)

pd.concat(
    [compact_results(default_results[name], name).head(2) for name in query_suite["scenario"]],
    ignore_index=True,
)


,scenario,rank,listing_id,city,price,score,source_ranks,signal_match,match_tier,retrieval_status,retrieval_evidence,summary
0,structured plus amenity,1,1157847051,Galt,459990,0.032266,"{'dense': 3, 'bm25': 1}",no extracted match,None,text_retrieval,dense #3; bm25 #1,"This 3-bed, 3-bath listing in Galt is listed at $459,990. excellent neighborhood in galt."
1,structured plus amenity,2,1157847482,Galt,469065,0.031754,"{'dense': 4, 'bm25': 2}",no extracted match,None,text_retrieval,dense #4; bm25 #2,"This 3-bed, 3-bath listing in Galt is listed at $469,065. excellent neighborhood in galt."
2,amenity retrieval,1,1151540840,Los Angeles,999000,0.039197,"{'dense': 1, 'bm25': 96}",pool,None,text_retrieval,dense #1; bm25 #96; signal: pool,"This 4-bed, 2-bath listing in Los Angeles is listed at $999,000. Highlights include solar and a private pool."
3,amenity retrieval,2,1174048031,Los Angeles,1849000,0.031545,{'dense': 6},pool,None,text_retrieval,dense #6; signal: pool,"This 3-bed, 2-bath listing in Los Angeles is listed at $1,849,000. Highlights include a pool and outdoor living."
4,condition and budget,1,1158639611,Riverside,669000,0.038032,"{'dense': 10, 'bm25': 76}",updated,None,text_retrieval,dense #10; bm25 #76; signal: updated,"This 3-bed, 2-bath listing in Riverside is listed at $669,000. Highlights include a covered patio and an open floor plan."
5,condition and budget,2,1173822687,Riverside,699900,0.02494,{'bm25': 57},updated,None,text_retrieval,bm25 #57; signal: updated,"This 5-bed, 2-bath listing in Riverside is listed at $699,900. Highlights include a backyard and a cabinetry."
6,property type and layout,1,1172142945,Irvine,1600000,0.027505,{'bm25': 30},open floor plan,None,text_retrieval,bm25 #30; signal: open floor plan,"This 4-bed, 4-bath listing in Irvine is listed at $1,600,000. Highlights include a pool and beach access."
7,property type and layout,2,1159246017,Irvine,1379000,0.045424,"{'dense': 3, 'bm25': 16}",open floor plan,None,text_retrieval,dense #3; bm25 #16; signal: open floor plan,"This 3-bed, 2-bath listing in Irvine is listed at $1,379,000. Highlights include a pool and an open floor plan."
8,location features,1,1169382643,Corona,705000,0.044462,"{'dense': 22, 'bm25': 3}",near schools,None,text_retrieval,dense #22; bm25 #3; signal: near schools,"This 3-bed, 2-bath listing in Corona is listed at $705,000. Highlights include beach access and full bathrooms."
9,location features,2,1144561342,Corona,1549000,0.039646,"{'dense': 20, 'bm25': 33}",near schools,None,text_retrieval,dense #20; bm25 #33; signal: near schools,"This 4-bed, 5-bath listing in Corona is listed at $1,549,000. Highlights include high ceilings and a gated community."


`source_ranks` shows which retrievers contributed a listing. `signal_match` distinguishes a source that was not used, a listing that matched but was not selected from a large signal set, and a listing without an extracted match.


---

## 4. Retrieval Component Comparison

The public default remains the two-worker service from Section 3. The controlled comparisons in this section use a separate serial service: dense, BM25, and signals therefore run in a fixed order, keeping retrieval component effects separate from thread scheduling.

The internal variants share the same parser, compliance snapshot, and hard eligibility. They are shown in retrieval order: Dense only, Dense + signals, then Hybrid RRF. Comparing their top-five overlap and warm latency shows whether BM25 or signals materially change the candidate mix. Section 8 measures ranking quality on manually judged dev labels.

In [7]:
serial_service = SearchService.from_active_snapshot(
    search_root=search_root,
    parallel_retrieval=False,
)

variants = {
    "dense_only": "Dense only",
    "dense_signal": "Dense + signals",
    "dense_bm25_signal_rrf": "Hybrid RRF",
}

comparison_suite = query_suite.loc[query_suite["scenario"] != "strict price sort"]
variant_order = list(variants.values())
variant_results = {}
variant_rows = []

for item in comparison_suite.to_dict("records"):
    for variant, label in variants.items():
        result = serial_service.search_experiment(item["query"], variant, top_k=5)
        variant_results[item["scenario"], variant] = result
        variant_rows.append({
            "scenario": item["scenario"],
            "variant": label,
            "retrieval_mode": result["meta"].get("retrieval_execution"),
            "returned": len(result["results"]),
            "signal_results": sum(row["signal_status"] == "selected" for row in result["results"]),
            "warm_latency_ms": result["meta"]["timings_ms"].get("total"),
            "hard_filter_ms": result["meta"]["timings_ms"].get("hard_filter"),
            "dense_ms": result["meta"]["timings_ms"].get("dense"),
            "bm25_ms": result["meta"]["timings_ms"].get("bm25"),
            "signals_ms": result["meta"]["timings_ms"].get("signals"),
            "retrieval_wall_ms": result["meta"]["timings_ms"].get("retrieval_wall"),
            "rrf_fusion_ms": result["meta"]["timings_ms"].get("rrf_fusion"),
            "signal_selection": result["meta"].get("signal_selection", {}).get("strategy"),
            "rrf_candidates": result["meta"].get("candidate_counts", {}).get("rrf_union"),
            "degraded_components": ", ".join(result["meta"]["degraded_components"]) or "none",
            "timings": result["meta"]["timings_ms"],
        })

variant_df = pd.DataFrame(variant_rows)
variant_df["variant"] = pd.Categorical(variant_df["variant"], categories=variant_order, ordered=True)
variant_df

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,scenario,variant,retrieval_mode,returned,signal_results,warm_latency_ms,hard_filter_ms,dense_ms,bm25_ms,signals_ms,retrieval_wall_ms,rrf_fusion_ms,signal_selection,rrf_candidates,degraded_components,timings
0,structured plus amenity,Dense only,serial,5,0,2731.03,7.78,2685.69,NaN,NaN,2685.70,0.01,not_used,5,none,"{'hard_filter': 7.78, 'dense': 2685.69, 'retrieval_wall': 2685.7, 'rrf_fusion': 0.01, 'total': 2731.03}"
1,structured plus amenity,Dense + signals,serial,5,0,59.17,8.80,6.64,NaN,2.02,8.68,0.01,no_query_signals,5,none,"{'hard_filter': 8.8, 'dense': 6.64, 'signals': 2.02, 'retrieval_wall': 8.68, 'rrf_fusion': 0.01, 'total': 59.17}"
2,structured plus amenity,Hybrid RRF,serial,5,0,50.10,6.19,6.31,1.01,1.83,9.17,0.01,no_query_signals,5,none,"{'hard_filter': 6.19, 'dense': 6.31, 'bm25': 1.01, 'signals': 1.83, 'retrieval_wall': 9.17, 'rrf_fusion': 0.01, 'total': 50.1}"
3,amenity retrieval,Dense only,serial,5,0,51.85,16.64,13.51,NaN,NaN,13.52,0.04,not_used,100,none,"{'hard_filter': 16.64, 'dense': 13.51, 'retrieval_wall': 13.52, 'rrf_fusion': 0.04, 'total': 51.85}"
4,amenity retrieval,Dense + signals,serial,5,5,74.15,23.68,21.12,NaN,6.67,27.87,0.16,dense_secondary,265,none,"{'hard_filter': 23.68, 'dense': 21.12, 'signals': 6.67, 'retrieval_wall': 27.87, 'rrf_fusion': 0.16, 'total': 74.15}"
5,amenity retrieval,Hybrid RRF,serial,5,5,83.08,21.64,18.65,14.70,5.13,38.55,0.18,dense_secondary,342,none,"{'hard_filter': 21.64, 'dense': 18.65, 'bm25': 14.7, 'signals': 5.13, 'retrieval_wall': 38.55, 'rrf_fusion': 0.18, 'total': 83.08}"
6,condition and budget,Dense only,serial,5,0,80.25,11.76,12.66,NaN,NaN,12.68,0.09,not_used,100,none,"{'hard_filter': 11.76, 'dense': 12.66, 'retrieval_wall': 12.68, 'rrf_fusion': 0.09, 'total': 80.25}"
7,condition and budget,Dense + signals,serial,5,5,61.15,9.45,11.49,NaN,3.62,15.15,0.38,all_matches,164,none,"{'hard_filter': 9.45, 'dense': 11.49, 'signals': 3.62, 'retrieval_wall': 15.15, 'rrf_fusion': 0.38, 'total': 61.15}"
8,condition and budget,Hybrid RRF,serial,5,5,60.09,8.75,10.57,3.85,3.35,17.80,0.41,all_matches,183,none,"{'hard_filter': 8.75, 'dense': 10.57, 'bm25': 3.85, 'signals': 3.35, 'retrieval_wall': 17.8, 'rrf_fusion': 0.41, 'total': 60.09}"
9,property type and layout,Dense only,serial,5,0,75.88,13.44,27.30,NaN,NaN,27.32,0.04,not_used,100,none,"{'hard_filter': 13.44, 'dense': 27.3, 'retrieval_wall': 27.32, 'rrf_fusion': 0.04, 'total': 75.88}"


In [8]:
overlap_rows = []

for scenario in comparison_suite["scenario"]:
    hybrid_ids = {item["listing_id"] for item in variant_results[scenario, "dense_bm25_signal_rrf"]["results"]}
    for variant, label in variants.items():
        result = variant_results[scenario, variant]
        result_ids = {item["listing_id"] for item in result["results"]}
        overlap_rows.append({
            "scenario": scenario,
            "variant": label,
            "top_5_jaccard_with_hybrid": len(hybrid_ids & result_ids) / len(hybrid_ids | result_ids),
        })

overlap_df = pd.DataFrame(overlap_rows)
component_summary = (
    variant_df.groupby("variant", as_index=False)
    .agg(
        mean_warm_latency_ms=("warm_latency_ms", "mean"),
        mean_signal_results=("signal_results", "mean"),
    )
    .merge(
        overlap_df.groupby("variant", as_index=False).agg(
            mean_top_5_jaccard_with_hybrid=("top_5_jaccard_with_hybrid", "mean")
        ),
        on="variant",
    )
)
component_summary["variant"] = pd.Categorical(component_summary["variant"], categories=variant_order, ordered=True)
component_summary.sort_values("variant")


,variant,mean_warm_latency_ms,mean_signal_results,mean_top_5_jaccard_with_hybrid
0,Dense only,600.860,0.0,0.372222
1,Dense + signals,67.286,4.0,0.372222
2,Hybrid RRF,69.476,4.0,1.000000


Use this comparison as a screening tool. A lower overlap means a component is changing retrieval, not that it is improving quality. Interpret quality through the manually judged dev comparison in Section 8.


In [9]:
review_scenario = "amenity retrieval"
review_rows = []

for variant, label in variants.items():
    for item in variant_results[review_scenario, variant]["results"]:
        review_rows.append({
            "variant": label,
            "rank": item["rank"],
            "listing_id": item["listing_id"],
            "price": item["price"],
            "source_ranks": item["source_ranks"],
            "signal_match": signal_display(item),
            "retrieval_status": item["retrieval_status"],
            "remark_excerpt": item["remarks_cleaned"][:240],
        })

review_df = pd.DataFrame(review_rows)
review_df["variant"] = pd.Categorical(review_df["variant"], categories=variant_order, ordered=True)
review_df.sort_values(["variant", "rank"])


,variant,rank,listing_id,price,source_ranks,signal_match,retrieval_status,remark_excerpt
0,Dense only,1,1151540840,999000,{'dense': 1},not used,text_retrieval,"this private pool home, situated in los angeles, boasts a 2 story structure with 4 bedrooms and 2 bathrooms. throughout its history, the property has undergone numerous updates, resulting in a modern living space. notably, it features p..."
1,Dense only,2,1173252181,2995000,{'dense': 2},not used,text_retrieval,"pool. accessory dwelling unit. 1 story. five bedrooms. under 3000000. fully renovated modern farmhouse in westchester featuring 6 rooms, 4.5 bathroom, two fireplaces, a chef's kitchen, and over 3000 square feet of thoughtfully designed ..."
2,Dense only,3,1174542960,575000,{'dense': 3},not used,text_retrieval,"pool. basement. 3 car garage. 7033 square feet lot. opportunities like this don't come along often in boyle heights. perched above the street on an elevated lot, this 3 bedroom, 2 bath home offers character, space, and tremendous potent..."
3,Dense only,4,1100752152,845000,{'dense': 4},not used,text_retrieval,owner says bring your offer. exquisite hollywood condominium home pool spa fitness center save those gym fees and more enjoy life in hollywood fashion chic and gorgeous designs well maintained and perfect for anyone searching for los an...
4,Dense only,5,1156430056,1200000,{'dense': 5},not used,text_retrieval,"los angeles single family residence r3 zoning water heater unit newly replaced december 2025 entire roof replacement done yr2020. the garage has been transformed into an awesome storage room held by the same family since 1975, 178 n mar..."
5,Dense + signals,1,1151540840,999000,{'dense': 1},pool,text_retrieval,"this private pool home, situated in los angeles, boasts a 2 story structure with 4 bedrooms and 2 bathrooms. throughout its history, the property has undergone numerous updates, resulting in a modern living space. notably, it features p..."
6,Dense + signals,2,1173252181,2995000,{'dense': 2},pool,text_retrieval,"pool. accessory dwelling unit. 1 story. five bedrooms. under 3000000. fully renovated modern farmhouse in westchester featuring 6 rooms, 4.5 bathroom, two fireplaces, a chef's kitchen, and over 3000 square feet of thoughtfully designed ..."
7,Dense + signals,3,1174542960,575000,{'dense': 3},pool,text_retrieval,"pool. basement. 3 car garage. 7033 square feet lot. opportunities like this don't come along often in boyle heights. perched above the street on an elevated lot, this 3 bedroom, 2 bath home offers character, space, and tremendous potent..."
8,Dense + signals,4,1100752152,845000,{'dense': 4},pool,text_retrieval,owner says bring your offer. exquisite hollywood condominium home pool spa fitness center save those gym fees and more enjoy life in hollywood fashion chic and gorgeous designs well maintained and perfect for anyone searching for los an...
9,Dense + signals,5,1174048031,1849000,{'dense': 6},pool,text_retrieval,"welcome home enjoy a quintessential california lifestyle with this spectacular and rare pool home that sits on a large, gated corner lot, boasting almost 8000 square feet, at the top of st. andrews place. privacy and pride of ownership ..."


---

## 5. Structured Filters and Price Sorting

Price sorting has two paths. A pure field request skips text retrieval and sorts the eligible set by price and `listing_id`. A request with positive soft preferences first groups complete signal matches, partial signal matches, text fallback candidates, and remaining eligible listings; each group is then sorted by price.


In [10]:
sort_query = "Find homes in Galt"
sort_rows = []

for sort_by in ["relevance", "price_asc", "price_desc"]:
    result = serial_service.search(sort_query, top_k=5, sort_by=sort_by)
    for item in result["results"]:
        sort_rows.append({
            "sort_by": sort_by,
            "rank": item["rank"],
            "listing_id": item["listing_id"],
            "price": item["price"],
            "match_tier": item.get("match_tier"),
            "signal_match": signal_display(item),
            "retrieval_status": item["retrieval_status"],
            "retrieval_evidence": item["retrieval_evidence"],
        })

pd.DataFrame(sort_rows)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

,sort_by,rank,listing_id,price,match_tier,signal_match,retrieval_status,retrieval_evidence
0,relevance,1,1159716277,469990,None,no extracted match,text_retrieval,dense #1; bm25 #4
1,relevance,2,1155116593,449990,None,no extracted match,text_retrieval,dense #2; bm25 #3
2,relevance,3,1157847482,469065,None,no extracted match,text_retrieval,dense #4; bm25 #2
3,relevance,4,1157847051,459990,None,no extracted match,text_retrieval,dense #3; bm25 #1
4,relevance,5,1168382149,575000,None,no extracted match,text_retrieval,dense #5; bm25 #5
5,price_asc,1,1155116593,449990,field_price,not used,field_sort,field sort
6,price_asc,2,1157847051,459990,field_price,not used,field_sort,field sort
7,price_asc,3,1157847482,469065,field_price,not used,field_sort,field sort
8,price_asc,4,1159716277,469990,field_price,not used,field_sort,field sort
9,price_asc,5,1168382149,575000,field_price,not used,field_sort,field sort


In [11]:
price_examples = {
    "pure field sort": serial_service.search("Show me the cheapest homes in Galt", top_k=5),
    "soft-preference price sort": serial_service.search(
        "Show me the cheapest homes in Los Angeles with a pool",
        top_k=5,
    ),
}

price_example_rows = []
for label, result in price_examples.items():
    for item in result["results"]:
        price_example_rows.append({
            "path": label,
            "rank": item["rank"],
            "listing_id": item["listing_id"],
            "price": item["price"],
            "match_tier": item.get("match_tier"),
            "signal_match": signal_display(item),
            "retrieval_evidence": item["retrieval_evidence"],
        })

pd.DataFrame(price_example_rows)


,path,rank,listing_id,price,match_tier,signal_match,retrieval_evidence
0,pure field sort,1,1155116593,449990,field_price,not used,field sort
1,pure field sort,2,1157847051,459990,field_price,not used,field sort
2,pure field sort,3,1157847482,469065,field_price,not used,field sort
3,pure field sort,4,1159716277,469990,field_price,not used,field sort
4,pure field sort,5,1168382149,575000,field_price,not used,field sort
5,soft-preference price sort,1,1153121002,218000,signal_complete,pool,signal: pool
6,soft-preference price sort,2,1155162318,295000,signal_complete,pool,signal: pool
7,soft-preference price sort,3,1151966538,299000,signal_complete,pool,signal: pool
8,soft-preference price sort,4,1152934624,299000,signal_complete,pool,signal: pool
9,soft-preference price sort,5,1113094863,299900,signal_complete,pool,signal: pool


---

## 6. Cross Encoder Rerank Experiment

This is a controlled serial comparison. The Cross Encoder reranks the same Hybrid RRF candidate window, while retrieval stays serial for both paths. Section 9 separately measures the default two-worker execution mode.

In [12]:
rerank_service = SearchService.from_active_snapshot(
    search_root=search_root,
    enable_cross_encoder=True,
    parallel_retrieval=False,
)
cross_encoder_cold_start = rerank_service.warm_up(include_cross_encoder=True)

cross_rows = []
cross_review_rows = []
cross_results = {}

for item in comparison_suite.to_dict("records"):
    scenario = item["scenario"]
    hybrid = serial_service.search_experiment(item["query"], "dense_bm25_signal_rrf", top_k=5)
    reranked = rerank_service.search_experiment(
        item["query"],
        "hybrid_cross_encoder",
        top_k=5,
    )
    cross_results[scenario] = reranked
    cross_rows.append({
        "scenario": scenario,
        "retrieval_mode": reranked["meta"].get("retrieval_execution"),
        "hybrid_warm_latency_ms": hybrid["meta"]["timings_ms"].get("total"),
        "cross_encoder_rerank_ms": reranked["meta"]["timings_ms"].get("cross_encoder_rerank"),
        "hybrid_cross_encoder_total_ms": reranked["meta"]["timings_ms"].get("total"),
        "incremental_rerank_ms": (
            reranked["meta"]["timings_ms"].get("total")
            - hybrid["meta"]["timings_ms"].get("total")
        ),
        "rrf_candidates": reranked["meta"].get("candidate_counts", {}).get("rrf_union"),
        "rerank_window": reranked["meta"].get("candidate_counts", {}).get("rerank_window"),
        "timings": reranked["meta"]["timings_ms"],
    })

    for variant, result in (("Hybrid RRF", hybrid), ("Hybrid + Cross Encoder", reranked)):
        for row in result["results"]:
            cross_review_rows.append({
                "scenario": scenario,
                "variant": variant,
                "rank": row["rank"],
                "pre_rerank_rank": row.get("pre_rerank_rank", row["rank"]),
                "listing_id": row["listing_id"],
                "price": row["price"],
                "retrieval_evidence": row["retrieval_evidence"],
                "cross_encoder_score": row.get("cross_encoder_score"),
            })

cross_encoder_cold_start
cross_encoder_latency = pd.DataFrame(cross_rows)
cross_encoder_review = pd.DataFrame(cross_review_rows)
display(cross_encoder_latency)
display(cross_encoder_review)

cross_encoder_examples = pd.DataFrame([
    {
        "rank": row["rank"],
        "listing_id": row["listing_id"],
        "summary": row["summary"],
        "remark_excerpt": row["remarks_cleaned"][:240],
    }
    for row in cross_results["amenity retrieval"]["results"]
])
cross_encoder_examples


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

,scenario,retrieval_mode,hybrid_warm_latency_ms,cross_encoder_rerank_ms,hybrid_cross_encoder_total_ms,incremental_rerank_ms,rrf_candidates,rerank_window,timings
0,structured plus amenity,serial,54.73,27.95,77.30,22.57,5,5,"{'hard_filter': 3.93, 'dense': 6.38, 'bm25': 0.98, 'signals': 1.83, 'retrieval_wall': 9.2, 'rrf_fusion': 0.01, 'cross_encoder_rerank': 27.95, 'total': 77.3}"
1,amenity retrieval,serial,63.92,142.28,223.50,159.58,342,50,"{'hard_filter': 21.18, 'dense': 17.71, 'bm25': 11.17, 'signals': 6.76, 'retrieval_wall': 35.72, 'rrf_fusion': 0.19, 'cross_encoder_rerank': 142.28, 'total': 223.5}"
2,condition and budget,serial,61.14,113.41,174.09,112.95,183,50,"{'hard_filter': 9.17, 'dense': 11.53, 'bm25': 3.14, 'signals': 1.58, 'retrieval_wall': 16.27, 'rrf_fusion': 0.18, 'cross_encoder_rerank': 113.41, 'total': 174.09}"
3,property type and layout,serial,73.00,123.85,195.37,122.37,283,50,"{'hard_filter': 10.24, 'dense': 11.99, 'bm25': 4.29, 'signals': 2.37, 'retrieval_wall': 18.7, 'rrf_fusion': 0.16, 'cross_encoder_rerank': 123.85, 'total': 195.37}"
4,location features,serial,65.05,137.65,220.10,155.05,185,50,"{'hard_filter': 10.09, 'dense': 13.15, 'bm25': 4.45, 'signals': 8.12, 'retrieval_wall': 25.76, 'rrf_fusion': 0.43, 'cross_encoder_rerank': 137.65, 'total': 220.1}"


,scenario,variant,rank,pre_rerank_rank,listing_id,price,retrieval_evidence,cross_encoder_score
0,structured plus amenity,Hybrid RRF,1,1,1155116593,449990,dense #1; bm25 #3,NaN
1,structured plus amenity,Hybrid RRF,2,2,1157847051,459990,dense #3; bm25 #1,NaN
2,structured plus amenity,Hybrid RRF,3,3,1157847482,469065,dense #4; bm25 #2,NaN
3,structured plus amenity,Hybrid RRF,4,4,1159716277,469990,dense #2; bm25 #4,NaN
4,structured plus amenity,Hybrid RRF,5,5,1168382149,575000,dense #5; bm25 #5,NaN
5,structured plus amenity,Hybrid + Cross Encoder,1,2,1157847051,459990,dense #3; bm25 #1,3.161113
6,structured plus amenity,Hybrid + Cross Encoder,2,3,1157847482,469065,dense #4; bm25 #2,3.138947
7,structured plus amenity,Hybrid + Cross Encoder,3,1,1155116593,449990,dense #1; bm25 #3,2.762884
8,structured plus amenity,Hybrid + Cross Encoder,4,4,1159716277,469990,dense #2; bm25 #4,2.500869
9,structured plus amenity,Hybrid + Cross Encoder,5,5,1168382149,575000,dense #5; bm25 #5,2.450682


,rank,listing_id,summary,remark_excerpt
0,1,1151540840,"This 4-bed, 2-bath listing in Los Angeles is listed at $999,000. Highlights include solar and a private pool.","this private pool home, situated in los angeles, boasts a 2 story structure with 4 bedrooms and 2 bathrooms. throughout its history, the property has undergone numerous updates, resulting in a modern living space. notably, it features p..."
1,2,1174048031,"This 3-bed, 2-bath listing in Los Angeles is listed at $1,849,000. Highlights include a pool and outdoor living.","welcome home enjoy a quintessential california lifestyle with this spectacular and rare pool home that sits on a large, gated corner lot, boasting almost 8000 square feet, at the top of st. andrews place. privacy and pride of ownership ..."
2,3,1158372360,"This 3-bed, 3-bath listing in Los Angeles is listed at $1,690,000. Highlights include a pool and an open floor plan.","set high in the hills of los angeles, this gated architectural pool home offers privacy, presence, and seamless indoor outdoor living. positioned on an expansive 10000 square feet lot, the property welcomes you with a spacious front cou..."
3,4,1174220082,"This 4-bed, 5-bath listing in Los Angeles is listed at $3,895,000. Highlights include a pool and a remodeled interior.","set among mature trees on an oversized, gated lot, 2216 e. live oak offers privacy, scale, and indoor outdoor living in one of los feliz's most sought after hillside settings. this spanish contemporary pool home opens with a formal entr..."
4,5,1172051387,"This 3-bed, 1-bath listing in Los Angeles is listed at $865,000. Highlights include a pool and solar.","a rare find on a tree lined street in south los angeles, this charming spanish bungalow offers an open floor plan with 3 spacious bedrooms and 1 generously sized bathroom. the bathroom features a newly installed bidet with warming seat ..."


---

## 7. Latency Summary

Section 3 reports the default two-worker service. Sections 4 through 6 are controlled serial runs, so their timings are useful for component and rerank comparisons but should not be read as the default latency profile. Section 9 compares execution modes on the frozen dev set.

In [13]:
timing_rows = []

for run, rows in (
    ("Default relevance", [row for row in default_rows if row["effective_sort"] == "relevance"]),
    ("Pure price sort", [row for row in default_rows if row["effective_sort"] != "relevance"]),
    ("Component variants", variant_rows),
    ("Hybrid + Cross Encoder", cross_rows),
):
    for row in rows:
        label = row.get("variant", run)
        for component, latency in row["timings"].items():
            if latency is not None:
                timing_rows.append({
                    "run": label,
                    "component": component,
                    "latency_ms": latency,
                })

latency_summary = (
    pd.DataFrame(timing_rows)
    .groupby(["run", "component"], as_index=False)
    .agg(
        mean_ms=("latency_ms", "mean"),
        p90_ms=("latency_ms", lambda values: np.percentile(values, 90)),
        p95_ms=("latency_ms", lambda values: np.percentile(values, 95)),
    )
)
latency_summary.sort_values(["run", "component"])


,run,component,mean_ms,p90_ms,p95_ms
0,Dense + signals,dense,13.486000,18.756,19.9380
1,Dense + signals,hard_filter,12.716000,18.668,21.1740
2,Dense + signals,retrieval_wall,18.580000,26.294,27.0820
3,Dense + signals,rrf_fusion,0.202000,0.372,0.3760
4,Dense + signals,signals,5.048000,9.214,10.0620
5,Dense + signals,total,67.286000,74.900,75.1500
6,Dense only,dense,550.500000,1622.334,2154.0120
7,Dense only,hard_filter,12.050000,15.360,16.0000
8,Dense only,retrieval_wall,550.516000,1622.348,2154.0240
9,Dense only,rrf_fusion,0.052000,0.086,0.0880


---

## 8. Judged Dev Relevance Comparison

The earlier sections are operational checks. This section uses manually graded dev qrels to compare ranking quality on the same pass-only snapshot.

The relevance variants were run with serial retrieval. Parallel execution does not change their candidates or relevance metrics; the execution comparison is reported separately in Section 9.

- Precision@5 and MRR@5 treat grades 2 and 3 as relevant.
- NDCG@5 preserves the full 0 to 3 relevance scale, so it rewards stronger matches near the top.
- The expanded qrels include blinded labels added after the document update and rerank-window pooling. The serial dev report below uses the 817-label pool available before final test evaluation. Section 8.3 records the later 849-label frozen manifest used for the held-out test.

In [14]:
results_path = Path("../data/processed/search_relevance_serial_dev_results.json")
relevance_results = json.loads(results_path.read_text())

variant_names = {
    "bm25_only": "BM25",
    "dense_only": "Dense",
    "dense_signal": "Dense + signals",
    "dense_bm25_signal_rrf": "Hybrid RRF",
    "hybrid_cross_encoder": "Hybrid + Cross Encoder",
}

metric_rows = []
for variant, name in variant_names.items():
    values = relevance_results["variants"][variant]["dev"]
    metric_rows.append({
        "variant": name,
        "precision_at_5": values["precision_at_5"],
        "ndcg_at_5": values["ndcg_at_5"],
        "mrr_at_5": values["mrr_at_5"],
        "p50_latency_ms": values["latency_ms"]["p50"],
        "p95_latency_ms": values["latency_ms"]["p95"],
    })

dev_metrics = pd.DataFrame(metric_rows)
dev_metrics.round(3)


,variant,precision_at_5,ndcg_at_5,mrr_at_5,p50_latency_ms,p95_latency_ms
0,BM25,0.721,0.605,0.836,66.82,206.99
1,Dense,0.586,0.499,0.815,83.63,144.88
2,Dense + signals,0.800,0.718,0.914,90.09,202.86
3,Hybrid RRF,0.843,0.808,0.976,87.37,407.32
4,Hybrid + Cross Encoder,0.907,0.877,0.964,305.87,671.37


In [15]:
rrf_metrics = dev_metrics.loc[dev_metrics["variant"] == "Hybrid RRF"].iloc[0]
cross_encoder_metrics = dev_metrics.loc[
    dev_metrics["variant"] == "Hybrid + Cross Encoder"
].iloc[0]

cross_encoder_change = pd.DataFrame([
    {
        "metric": "Precision@5",
        "Hybrid RRF": rrf_metrics["precision_at_5"],
        "Cross Encoder": cross_encoder_metrics["precision_at_5"],
        "change": cross_encoder_metrics["precision_at_5"] - rrf_metrics["precision_at_5"],
    },
    {
        "metric": "NDCG@5",
        "Hybrid RRF": rrf_metrics["ndcg_at_5"],
        "Cross Encoder": cross_encoder_metrics["ndcg_at_5"],
        "change": cross_encoder_metrics["ndcg_at_5"] - rrf_metrics["ndcg_at_5"],
    },
    {
        "metric": "MRR@5",
        "Hybrid RRF": rrf_metrics["mrr_at_5"],
        "Cross Encoder": cross_encoder_metrics["mrr_at_5"],
        "change": cross_encoder_metrics["mrr_at_5"] - rrf_metrics["mrr_at_5"],
    },
    {
        "metric": "P95 latency (ms)",
        "Hybrid RRF": rrf_metrics["p95_latency_ms"],
        "Cross Encoder": cross_encoder_metrics["p95_latency_ms"],
        "change": cross_encoder_metrics["p95_latency_ms"] - rrf_metrics["p95_latency_ms"],
    },
])

cross_encoder_change.round(3)


,metric,Hybrid RRF,Cross Encoder,change
0,Precision@5,0.843,0.907,0.064
1,NDCG@5,0.808,0.877,0.069
2,MRR@5,0.976,0.964,-0.012
3,P95 latency (ms),407.320,671.370,264.050


With a rerank window of 50, the Cross Encoder improves Precision@5 and graded ranking quality over Hybrid RRF, while MRR@5 is slightly lower. It remains the default relevance path; if reranking is unavailable at runtime, the service returns the Hybrid RRF order with a degraded-component marker.

### 8.1 Query-Level Error Analysis

Aggregate metrics can hide where reranking helps or hurts. The table below compares the largest gains and declines in `NDCG@5` between Hybrid RRF and the adjusted Cross Encoder.

- Grades are ordered by returned rank, from first to fifth.
- Positive values mean the Cross Encoder improved graded ranking quality.
- This is a focused review list, not another aggregate metric.


In [16]:
query_path = Path("../data/processed/search_relevance_queries.json")
query_text = {
    item["id"]: item["query"]
    for item in json.loads(query_path.read_text())["items"]
}

rrf_by_query = {
    row["query_id"]: row
    for row in relevance_results["variants"]["dense_bm25_signal_rrf"]["dev"]["per_query"]
}
cross_encoder_by_query = {
    row["query_id"]: row
    for row in relevance_results["variants"]["hybrid_cross_encoder"]["dev"]["per_query"]
}

error_rows = []
for query_id, rrf_row in rrf_by_query.items():
    cross_encoder_row = cross_encoder_by_query[query_id]
    error_rows.append({
        "query_id": query_id,
        "query": query_text[query_id],
        "rrf_ndcg_at_5": rrf_row["ndcg_at_5"],
        "cross_encoder_ndcg_at_5": cross_encoder_row["ndcg_at_5"],
        "ndcg_change": cross_encoder_row["ndcg_at_5"] - rrf_row["ndcg_at_5"],
        "rrf_grades": rrf_row["grades"],
        "cross_encoder_grades": cross_encoder_row["grades"],
        "rrf_listing_ids": rrf_row["listing_ids"],
        "cross_encoder_listing_ids": cross_encoder_row["listing_ids"],
    })

error_analysis = pd.DataFrame(error_rows)
largest_declines = error_analysis.nsmallest(4, "ndcg_change")
largest_gains = error_analysis.nlargest(4, "ndcg_change")
error_review = pd.concat([largest_declines, largest_gains]).drop_duplicates("query_id")

error_review = error_review.sort_values("ndcg_change").reset_index(drop=True)
error_review[[
    "query_id",
    "query",
    "rrf_ndcg_at_5",
    "cross_encoder_ndcg_at_5",
    "ndcg_change",
    "rrf_grades",
    "cross_encoder_grades",
    "rrf_listing_ids",
    "cross_encoder_listing_ids",
]].round(3)


,query_id,query,rrf_ndcg_at_5,cross_encoder_ndcg_at_5,ndcg_change,rrf_grades,cross_encoder_grades,rrf_listing_ids,cross_encoder_listing_ids
0,rel_013,"Move-in ready homes in Pasadena, not a fixer upper",1.000,0.661,-0.339,"[3, 3, 3, 3, 3]","[0, 3, 3, 3, 3]","[1156818573, 1158630286, 1166340510, 1155216098, 1159840790]","[1173676886, 1174696130, 1156818573, 1157094872, 1173221904]"
1,rel_020,Homes with an attached garage and EV charging,0.703,0.488,-0.214,"[3, 0, 3, 2, 3]","[1, 3, 3, 0, 2]","[1156103461, 1157597344, 1150830622, 1155391245, 1171677032]","[1156906171, 1159736466, 1150830622, 1173551396, 1151347592]"
2,rel_011,Single story homes in Palm Springs with a pool,0.765,0.564,-0.201,"[3, 2, 3, 3, 1]","[3, 0, 2, 1, 3]","[1155644840, 1155413077, 1159936379, 1169431021, 1166289451]","[1159936379, 1151093768, 1174049140, 1166289451, 1155657336]"
3,rel_018,Homes with mountain views and a fireplace,1.000,0.817,-0.183,"[3, 3, 3, 3, 3]","[3, 1, 3, 3, 3]","[1150169857, 1159248292, 1152539025, 1158602690, 1135839255]","[1156581911, 1170059438, 1153028604, 1159248292, 1172318167]"
4,rel_005,Detached homes in Anaheim with a private backyard,0.277,0.677,0.399,"[0, 1, 3, 0, 2]","[3, 3, 2, 0, 0]","[1173188557, 1156040790, 1109048665, 1157525681, 1155414684]","[1144187364, 1158365966, 1158244131, 1173219157, 1173039936]"
5,rel_021,Homes in walkable locations near restaurants,0.524,1.000,0.476,"[3, 1, 2, 2, 1]","[3, 3, 3, 3, 3]","[1158658024, 1173067567, 1174212362, 1174682803, 1152541391]","[1157929787, 1171961103, 1158736950, 1161133373, 1156469041]"
6,rel_025,Homes with a pool and central air,0.476,1.000,0.524,"[3, 2, 1, 1, 0]","[3, 3, 3, 3, 3]","[1159158818, 1158583082, 1174624530, 1170882967, 1159849163]","[1170902190, 1159158818, 1158478121, 1159876697, 1159777653]"
7,rel_012,Homes in Chino Hills with a large lot,0.367,0.946,0.579,"[3, 0, 0, 0, 0]","[3, 3, 3, 1, 3]","[1169383587, 1144149606, 1158274423, 1159448009, 1159700598]","[1169383587, 1157701459, 1169831022, 1172852548, 1174127344]"


### 8.2 Rerank Window Trade-off

The rerank window is the number of RRF candidates scored by the Cross Encoder before returning the top five. It changes reranking cost, not the upstream retrieval sources.

The three windows below use the same snapshot, dev split, and expanded qrels. New candidates from the smaller windows were blinded and labeled before this comparison.


In [17]:
window_results_path = Path("../data/processed/search_relevance_rerank_window_dev_results.json")
window_results = json.loads(window_results_path.read_text())

window_rows = []
for window, values in window_results["windows"].items():
    metrics = values["dev"]
    window_rows.append({
        "rerank_window": int(window),
        "precision_at_5": metrics["precision_at_5"],
        "ndcg_at_5": metrics["ndcg_at_5"],
        "mrr_at_5": metrics["mrr_at_5"],
        "p50_latency_ms": metrics["latency_ms"]["p50"],
        "p95_latency_ms": metrics["latency_ms"]["p95"],
    })

rerank_window_metrics = (
    pd.DataFrame(window_rows)
    .sort_values("rerank_window")
    .reset_index(drop=True)
)
rerank_window_metrics.round(3)


,rerank_window,precision_at_5,ndcg_at_5,mrr_at_5,p50_latency_ms,p95_latency_ms
0,20,0.871,0.842,0.982,202.02,520.11
1,50,0.907,0.877,0.964,282.74,583.51
2,100,0.893,0.872,0.946,441.80,686.57


On this dev set, a window of 50 gives the strongest `Precision@5` and `NDCG@5`, while remaining materially faster than 100. A window of 20 is fastest but leaves quality on the table. The default relevance configuration now uses a window of 50. The Cross Encoder remains independently measurable through the internal experiment variants.


### 8.3 Final Held-out Test

The default configuration was frozen before this run: Hybrid + Cross Encoder, rerank window 50, and two-worker retrieval parallelism. Only this default path was evaluated on the test split. The result is reported once and was not used for further tuning.

In [18]:
final_test_path = Path("../data/processed/search_relevance_final_test_results.json")
final_test_results = json.loads(final_test_path.read_text())
final_test = final_test_results["variants"]["hybrid_cross_encoder"]["test"]

final_test_metrics = pd.DataFrame([{
    "variant": "Hybrid + Cross Encoder",
    "queries_evaluated": final_test["queries_evaluated"],
    "parallel_retrieval": final_test_results["parallel_retrieval"],
    "retrieval_workers": final_test_results["retrieval_workers"],
    "precision_at_5": final_test["precision_at_5"],
    "ndcg_at_5": final_test["ndcg_at_5"],
    "mrr_at_5": final_test["mrr_at_5"],
    "p50_latency_ms": final_test["latency_ms"]["p50"],
    "p95_latency_ms": final_test["latency_ms"]["p95"],
}])
final_test_metrics.round(3)

,variant,queries_evaluated,parallel_retrieval,retrieval_workers,precision_at_5,ndcg_at_5,mrr_at_5,p50_latency_ms,p95_latency_ms
0,Hybrid + Cross Encoder,12,True,2,0.867,0.901,0.917,279.58,535.72


The frozen default retained strong top-five quality on the held-out set. This is a snapshot-specific result from 12 queries, so it is a final project baseline rather than a production SLA or a substitute for a broader online evaluation.

---

## 9. Serial and Parallel Retrieval

This section isolates execution policy rather than ranking logic. The same frozen dev queries, qrels, snapshot, and rerank window are used for serial retrieval, two workers, and three workers.

- Quality metrics and returned top-five IDs must stay unchanged.
- retrieval_wall measures the elapsed retrieval path, while total latency also includes Cross Encoder inference.
- These are warm local runs, so the table is useful for choosing a default configuration rather than defining a production SLA.

In [19]:
execution_runs = [
    {
        "mode": "serial",
        "workers": 1,
        "path": Path("../data/processed/search_relevance_serial_dev_results.json"),
    },
    {
        "mode": "parallel",
        "workers": 2,
        "path": Path("../data/processed/search_relevance_parallel_2_workers_dev_results.json"),
    },
    {
        "mode": "parallel",
        "workers": 3,
        "path": Path("../data/processed/search_relevance_parallel_dev_results.json"),
    },
]

execution_rows = []
serial_ids = None

for run in execution_runs:
    payload = json.loads(run["path"].read_text())
    metrics = payload["variants"]["hybrid_cross_encoder"]["dev"]
    result_ids = [row["listing_ids"] for row in metrics["per_query"]]

    if serial_ids is None:
        serial_ids = result_ids

    retrieval_wall = metrics["component_latency_ms"]["retrieval_wall"]
    execution_rows.append({
        "mode": run["mode"],
        "workers": run["workers"],
        "same_top_5_as_serial": result_ids == serial_ids,
        "precision_at_5": metrics["precision_at_5"],
        "ndcg_at_5": metrics["ndcg_at_5"],
        "mrr_at_5": metrics["mrr_at_5"],
        "p50_retrieval_wall_ms": retrieval_wall["p50"],
        "p95_retrieval_wall_ms": retrieval_wall["p95"],
        "p50_total_ms": metrics["latency_ms"]["p50"],
        "p95_total_ms": metrics["latency_ms"]["p95"],
    })

execution_comparison = pd.DataFrame(execution_rows)
execution_comparison.round(3)

,mode,workers,same_top_5_as_serial,precision_at_5,ndcg_at_5,mrr_at_5,p50_retrieval_wall_ms,p95_retrieval_wall_ms,p50_total_ms,p95_total_ms
0,serial,1,True,0.907,0.877,0.964,30.22,363.01,305.87,671.37
1,parallel,2,True,0.907,0.877,0.964,38.59,317.39,279.06,562.88
2,parallel,3,True,0.907,0.877,0.964,46.25,342.34,326.12,621.92


The two-worker configuration preserves the full ranked output and all judged metrics. It produced the best warm end-to-end P50 and P95 in this comparison, while three workers showed more local contention. The service therefore defaults to two workers; serial retrieval remains available for controlled experiments through parallel_retrieval=False or --serial-retrieval in the evaluation script.

---

## 10. Review Notes

This notebook now answers both operational and judged-quality questions:

- Does the service load one coherent pass-only snapshot?
- Does the default two-worker path preserve hard-filter and retrieval behavior?
- Do BM25 and signals add distinct candidates without causing failures?
- Do pure price sorting and soft-preference price tiers follow different, explainable paths?
- Does Cross Encoder change the same Hybrid candidate window at an acceptable incremental cost?
- Does the judged dev comparison support the default Cross Encoder quality-latency trade-off?
- Does the parallel execution mode preserve ranking output while improving the local latency profile?